<a href="https://colab.research.google.com/github/03sarath/evidentlyAI-llm-eval/blob/main/LLM_evaluation_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM regression testing workflow

In [ ]:
!pip install 'evidently[llm]==0.7.23'

In [ ]:
import pandas as pd
import numpy as np
import requests
from io import BytesIO

To run open-source evaluations:

In [ ]:
from evidently import Dataset
from evidently import DataDefinition
from evidently import Report
from evidently.presets import TextEvals
from evidently.descriptors import *

**Optional**: To work with Evidently Cloud:

In [ ]:
from evidently.ui.workspace import CloudWorkspace

# Prepare a dataset

Get an example dataset. You can also download and import the CSV file directly ([Link](https://github.com/evidentlyai/evidently/blob/main/examples/how_to_questions/chat_df.csv)).

In [ ]:
response = requests.get("https://psitron.s3.ap-southeast-1.amazonaws.com/genai-materials/chat_df.csv")
csv_content = BytesIO(response.content)

Read the CSV content into a DataFrame. Parse dates and set conversation "start_time" as index.

In [ ]:
assistant_logs = pd.read_csv(csv_content, index_col=0, parse_dates=['start_time', 'end_time'])
assistant_logs.index = assistant_logs.start_time
assistant_logs.index.rename('index', inplace=True)

Preview:

In [ ]:
pd.set_option('display.max_colwidth', None)

In [ ]:
assistant_logs.head(3)

# Run evaluations

Prep: map your input data columns with `DataDefinition`. Optional, but recommended.

In [ ]:
data_definition = DataDefinition(
    timestamp='start_time',
    datetime_columns=['end_time'],
    text_columns=['question', 'response'],
    categorical_columns=['organization', 'model_ID', 'region', 'environment', 'feedback'],
)

## Basic example

Run the first evaluation by checking the chatbot response length. You will use the `TextLength()` descriptor. This will return an absolute count for the number of symbols in each text. You can also check `SentenceCount()`, `WordCount()`, etc.

To run the evaluation for the first 100 conversations: add the descriptor to a `Dataset`, then get a summary Report with the `TextEvals` preset:

In [ ]:
eval_dataset = Dataset.from_pandas(
    assistant_logs[:100],
    data_definition=data_definition,
    descriptors=[
        TextLength("response", alias="Response Length"),
    ]
)

report = Report([
    TextEvals()
])

my_eval = report.run(eval_dataset, None)
my_eval

You can also do a side-by-side comparison for two datasets:

In [ ]:
eval_dataset = Dataset.from_pandas(
    assistant_logs[50:100],
    data_definition=data_definition,
    descriptors=[
        TextLength("response", alias="Response Length"),
    ]
)

ref_dataset = Dataset.from_pandas(
    assistant_logs[:50],
    data_definition=data_definition,
    descriptors=[
        TextLength("response", alias="Response Length"),
    ]
)

report = Report([
    TextEvals()
])

my_eval = report.run(eval_dataset, ref_dataset)
my_eval

Let's look at other evaluation methods one by one. You can later combine multiple descriptors in a single Report.

## Text patterns

You can use regular expressions to check text patterns. For example, check the presence of competitor mentions, topical words, etc.

Let's check for responses that contain words related to compensation. This will automatically account for inflected and variant words. This descriptor returns True/False for **pattern match**.

In [ ]:
eval_dataset = Dataset.from_pandas(
    assistant_logs[:100],
    data_definition=data_definition,
    descriptors=[
        IncludesWords("response",
                      words_list=['salary', 'benefits', 'payroll'],
                      alias="Mention Compensation"),
    ]
)

report = Report([
    TextEvals()
])

my_eval = report.run(eval_dataset, None)
my_eval

Other examples: `Contains(items=[])`, `BeginsWith(prefix="")`, custom `RegExp(reg_exp=r"")`, etc.

## Model-based scoring

You can use pre-trained machine learning models to score your text data.

**Sentiment**. You can use built-in models like `Sentiment()`. This will return a sentiment score from -1 (very negative) to 1 (very positive).

In [ ]:
eval_dataset = Dataset.from_pandas(
    assistant_logs[:100],
    data_definition=data_definition,
    descriptors=[
        Sentiment("response", alias="Sentiment"),
    ]
)

report = Report([
    TextEvals()
])

my_eval = report.run(eval_dataset, None)
my_eval

You can also use models from HuggingFace. This will download the models to score your data locally.

**Toxicity**. You can use a pre-selected toxicity model using `HuggingFaceToxicity()` descriptor. This will return the predicted toxicity score between 0 to 1.

**Neutral emotion**. You can call a named custom model from HuggingFace. For example, let's use the `SamLowe/roberta-base-go_emotions` model and get a score from 0 to 1 for "neutral" label to see if responses convey neutral emotion.

In [ ]:
eval_dataset = Dataset.from_pandas(
    assistant_logs[:100],
    data_definition=data_definition,
    descriptors=[
        HuggingFaceToxicity("response", alias="Toxicity"),
        HuggingFace("response",
                    model="SamLowe/roberta-base-go_emotions",
                    params={"label": "neutral"},
                    alias="Response Neutrality"),
    ]
)

report = Report([
    TextEvals()
])

my_eval = report.run(eval_dataset, None)
my_eval

To see the raw scores for each row:

In [ ]:
eval_dataset.as_dataframe()

See docs on using HuggingFace models as descriptors: https://docs.evidentlyai.com/metrics/customize_hf_descriptor